In [11]:
import tensorflow as tf
import tensorflow_probability as tfp
import gym
import numpy as np
import pandas as pd

In [48]:
class MAMLPPOAgent:
    def __init__(self, state_dim, action_dim, hidden_dim, learning_rate, gamma, lambda_):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.lambda_ = lambda_

        self.actor_network = self.build_actor_network()
        self.critic_network = self.build_critic_network()

    def build_actor_network(self):
        inputs = tf.keras.layers.Input(shape=(self.state_dim,))
        x = tf.keras.layers.Dense(self.hidden_dim, activation='relu')(inputs)
        outputs = tf.keras.layers.Dense(self.action_dim, activation='tanh')(x)
        model = tf.keras.Model(inputs=inputs, outputs=outputs)
        return model

    def build_critic_network(self):
        inputs = tf.keras.layers.Input(shape=(self.state_dim,))
        x = tf.keras.layers.Dense(self.hidden_dim, activation='relu')(inputs)
        outputs = tf.keras.layers.Dense(1)(x)
        model = tf.keras.Model(inputs=inputs, outputs=outputs)
        return model

    def compute_log_prob(self, states, actions):
        logits = self.actor_network(states)
        dist = tfp.distributions.MultivariateNormalDiag(loc=logits, scale_diag=tf.ones_like(logits))
        log_prob = dist.log_prob(actions)
        return log_prob

    def compute_advantages(self, rewards, values, next_values):
        deltas = rewards + self.gamma * next_values - values
        advantages = tf.zeros_like(rewards)
        advantage = 0.0
        for t in reversed(range(len(rewards))):
            advantage = self.gamma * self.lambda_ * advantage + deltas[t]
            advantages[t] = advantage
        return advantages

    def train_step(self, states, actions, rewards, values, next_values):
        log_prob = self.compute_log_prob(states, actions)
        advantages = self.compute_advantages(rewards, values, next_values)

        actor_loss = -tf.reduce_mean(log_prob * advantages)
        critic_loss = tf.reduce_mean(tf.square(values - rewards))

        total_loss = actor_loss + critic_loss
        gradients = tf.gradients(total_loss, self.actor_network.trainable_variables + self.critic_network.trainable_variables)
        optimizer = tf.keras.optimizers.Adam(learning_rate=self.learning_rate)
        optimizer.apply_gradients(zip(gradients, self.actor_network.trainable_variables + self.critic_network.trainable_variables))


In [49]:
class StockTradingEnv(gym.Env):
    def __init__(self, data, window_size, initial_balance):
        self.data = data
        self.window_size = window_size
        self.initial_balance = initial_balance
        self.current_step = 0

        self.action_space = gym.spaces.Discrete(3)
        self.observation_space = gym.spaces.Box(low=0.0, high=np.inf, shape=(window_size, data.shape[1]))

    def reset(self):
        self.current_step = 0
        self.balance = self.initial_balance
        self.net_worth = self.initial_balance
        self.inventory = []
        return self._next_observation()

    def step(self, action):
        if action == 0:
            if self.data[self.current_step, 3] > 0:
                self.inventory.append(self.data[self.current_step, 3])
                self.balance -= self.data[self.current_step, 3]
        elif action == 1:
            if len(self.inventory) > 0:
                self.balance += self.data[self.current_step, 3]
                self.inventory.pop(0)
        self.net_worth = self.balance + sum(self.inventory)

        done = self.current_step == len(self.data) - 1
        obs = self._next_observation()
        reward = self.net_worth - self.initial_balance

        self.current_step += 1
        return obs, reward, done, {}

    def _next_observation(self):
        end = min(self.current_step + self.window_size, len(self.data))
        obs = self.data[self.current_step:end, :]
        return obs

In [50]:
def fetch_data():
    df = pd.read_csv('./datas/samsung.csv')
    data = df.drop(columns=['Date'])
    return data.values

In [53]:
data = fetch_data()

# Set the hyperparameters
state_dim = data.shape[1]  # Number of features in the state (e.g., open, high, low, close, volume)
action_dim = 3  # Number of actions (buy, sell, hold)
hidden_dim = 64  # Number of hidden units in the actor and critic networks
learning_rate = 0.001  # Learning rate for the optimizer
gamma = 0.99  # Discount factor
lambda_ = 0.95  # GAE parameter
window_size = 10  # Size of the observation window
initial_balance = 1000000  # Initial balance for the agent

# Initialize the environment
env = StockTradingEnv(data, window_size, initial_balance)

# Initialize the agent
agent = MAMLPPOAgent(state_dim, action_dim, hidden_dim, learning_rate, gamma, lambda_)

# Training loop
num_episodes = 1000
for episode in range(num_episodes):
    state = env.reset()
    done = False
    episode_reward = 0

    states, actions, rewards, values, next_values = [], [], [], [], []
    while not done:
        action = agent.actor_network.predict(state)[0]
        action = np.argmax(action)

        next_state, reward, done, _ = env.step(action)

        states.append(state)
        actions.append(action)
        rewards.append(reward)
        values.append(agent.critic_network.predict(state)[0])
        next_values.append(agent.critic_network.predict(next_state)[0])

        state = next_state
        episode_reward += reward

    next_values.append(agent.critic_network.predict(next_state)[0])
    advantages = agent.compute_advantages(np.array(rewards), np.array(values), np.array(next_values))

    agent.train_step(np.array(states), np.array(actions), np.array(rewards), np.array(values), np.array(next_values))

    print(f"Episode {episode+1}: Reward = {episode_reward}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step

ValueError: operands could not be broadcast together with shapes (246,245) (245,1) 